In [ ]:
from openff.interchange import Interchange
import openff

from glob import glob
import re
from pathlib import Path

# Load NAGL for Parametrisation

In [ ]:
from openff.toolkit import Molecule, Topology, ForceField
from openff.interchange import Interchange

from openff.toolkit.utils.nagl_wrapper import (
            NAGLToolkitWrapper as _NAGLToolkitWrapper,
        )

_has_nagl = _NAGLToolkitWrapper.is_available()
from openff.nagl_models import get_models_by_type as _get_models_by_type

_models = _get_models_by_type("am1bcc")
from pathlib import Path
try:
    # Find the most recent AM1-BCC release candidate.
    _nagl = _NAGLToolkitWrapper()
    # choose openff-gnn-am1bcc-0.1.0-rc.3.pt model
    target_name = "openff-gnn-am1bcc-0.1.0-rc.3.pt"
    _nagl_model = next((m for m in _models if m.name == target_name), None)
except:
    _nagl_model = None

In [ ]:
_models

In [ ]:
_nagl_model

In [ ]:
# main parametrisation function

def prepare_ligand(sdf_file, version="openff_2_2_0_rc_1_nagl_0_1_0_rc3"):
    basename = Path(sdf_file).stem  # e.g. "compound_17_rb_no_ole_manual_build"

    match = re.match(r"(compound_\d+)_(.+)", basename)
    if not match:
        raise ValueError(
            f"{basename!r} doesn't match 'compound_<number>_<suffix>'"
        )b
    compound, suffix = match.groups()  # "compound_17", "rb_no_ole_manual_build"

    out_name = f"{compound}_{version}_{suffix}"
    out_dir = Path("parametrised_ligands")
    out_dir.mkdir(exist_ok=True)

    molecule = Molecule.from_file(sdf_file)

    # molecule.generate_conformers(n_conformers=1)
    _nagl.assign_partial_charges(molecule, partial_charge_method=_nagl_model)
    charge_from_molecules = [molecule]

    topology = molecule.to_topology()
    forcefield = ForceField("openff_unconstrained-2.2.0-rc1.offxml")

    interchange = Interchange.from_smirnoff(
        force_field=forcefield,
        topology=topology,
        charge_from_molecules=charge_from_molecules,
    )

    interchange.to_prmtop(out_dir / f"{out_name}.prm7")
    interchange.to_inpcrd(out_dir / f"{out_name}.rst7")
    interchange.to_pdb(out_dir / f"{out_name}.pdb")


# Parametrise molecules

In [ ]:
# Get all SDF files in the raw ligands directory
sdf_files = glob("../../data/ligands/*.sdf")

# Add each SDF file to the list of molecules to parametrize
for sdf_file in sdf_files:
    print(f"Parametrising ligand: {sdf_file}")
    prepare_ligand(sdf_file)